# 00.- Librerias

In [2]:
import mysql.connector
from mysql.connector import Error
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# 01.- Viajeros_pernoctaciones_por_comunidades_autónomas_provincias

## 01.1.- Importar fichero csv

In [3]:
df_comunidades = pd.read_csv("2026_06_15_Viajeros_pernoctaciones_por_comunidades_autónomas_provincias.csv", sep=';')

In [94]:
df_comunidades

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia: Nivel 1,Residencia: Nivel 2,Periodo,Total
0,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M04,10.211.051
1,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M03,8.270.160
2,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M02,6.594.807
3,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M01,5.948.424
4,Total Nacional,NaN,NaN,Viajero,Total,NaN,2025M12,6.937.385
...,...,...,...,...,...,...,...,...
137755,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M05,1.305
137756,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M04,1.668
137757,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M03,1.673
137758,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M02,1.487


## 01.2.- Transformación

In [95]:
# Tipos de datos e información general
df_comunidades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137760 entries, 0 to 137759
Data columns (total 8 columns):
 #   Column                            Non-Null Count   Dtype 
---  ------                            --------------   ----- 
 0   Totales Territoriales             137760 non-null  object
 1   Comunidades y Ciudades Autónomas  135792 non-null  object
 2   Provincias                        98400 non-null   object
 3   Viajeros y pernoctaciones         137760 non-null  object
 4   Residencia: Nivel 1               137760 non-null  object
 5   Residencia: Nivel 2               91840 non-null   object
 6   Periodo                           137760 non-null  object
 7   Total                             137750 non-null  object
dtypes: object(8)
memory usage: 8.4+ MB


In [96]:
# Convertir columna Total de object a numerica

# 1.- Quitar el punto (.) como separador de miles
df_comunidades["Total"] = df_comunidades["Total"].str.replace(".", "", regex=False)

# 2.- Reemplazar el "0" por NaN
df_comunidades["Total"] = df_comunidades["Total"].replace({"0": np.nan})

# 3.- Convertir a tipo integer
df_comunidades["Total"] = pd.to_numeric(df_comunidades["Total"]).astype("Int64")



In [97]:
df_comunidades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137760 entries, 0 to 137759
Data columns (total 8 columns):
 #   Column                            Non-Null Count   Dtype 
---  ------                            --------------   ----- 
 0   Totales Territoriales             137760 non-null  object
 1   Comunidades y Ciudades Autónomas  135792 non-null  object
 2   Provincias                        98400 non-null   object
 3   Viajeros y pernoctaciones         137760 non-null  object
 4   Residencia: Nivel 1               137760 non-null  object
 5   Residencia: Nivel 2               91840 non-null   object
 6   Periodo                           137760 non-null  object
 7   Total                             137255 non-null  Int64 
dtypes: Int64(1), object(7)
memory usage: 8.5+ MB


In [98]:
df_comunidades['Comunidades y Ciudades Autónomas'].unique()


array([nan, '01 Andalucía', '02 Aragón', '03 Asturias, Principado de',
       '04 Balears, Illes', '05 Canarias', '06 Cantabria',
       '07 Castilla y León', '08 Castilla - La Mancha', '09 Cataluña',
       '10 Comunitat Valenciana', '11 Extremadura', '12 Galicia',
       '13 Madrid, Comunidad de', '14 Murcia, Región de',
       '15 Navarra, Comunidad Foral de', '16 País Vasco', '17 Rioja, La',
       '18 Ceuta', '19 Melilla'], dtype=object)

In [99]:
# Crear una nueva columna con el codigo de comunidad
df_comunidades['cod_comunidad'] = (
    df_comunidades['Comunidades y Ciudades Autónomas']
    .str.strip()
    .str[:2]
)


# Crear una nueva columna con el nombre de la comunidad
df_comunidades['nombre_comunidad'] = (
    df_comunidades['Comunidades y Ciudades Autónomas']
    .str.strip()
    .str[3:]
    .str.strip()
)

In [100]:
df_comunidades

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia: Nivel 1,Residencia: Nivel 2,Periodo,Total,cod_comunidad,nombre_comunidad
0,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M04,10211051,NaN,NaN
1,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M03,8270160,NaN,NaN
2,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M02,6594807,NaN,NaN
3,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M01,5948424,NaN,NaN
4,Total Nacional,NaN,NaN,Viajero,Total,NaN,2025M12,6937385,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
137755,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M05,1305,19,Melilla
137756,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M04,1668,19,Melilla
137757,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M03,1673,19,Melilla
137758,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M02,1487,19,Melilla


In [101]:
df_comunidades['Provincias'].unique()

array([nan, '04 Almería', '11 Cádiz', '14 Córdoba', '18 Granada',
       '21 Huelva', '23 Jaén', '29 Málaga', '41 Sevilla', '22 Huesca',
       '44 Teruel', '50 Zaragoza', '33 Asturias', '07 Balears, Illes',
       '35 Palmas, Las', '38 Santa Cruz de Tenerife', '39 Cantabria',
       '05 Ávila', '09 Burgos', '24 León', '34 Palencia', '37 Salamanca',
       '40 Segovia', '42 Soria', '47 Valladolid', '49 Zamora',
       '02 Albacete', '13 Ciudad Real', '16 Cuenca', '19 Guadalajara',
       '45 Toledo', '08 Barcelona', '17 Girona', '25 Lleida',
       '43 Tarragona', '03 Alicante/Alacant', '12 Castellón/Castelló',
       '46 Valencia/València', '06 Badajoz', '10 Cáceres', '15 Coruña, A',
       '27 Lugo', '32 Ourense', '36 Pontevedra', '28 Madrid', '30 Murcia',
       '31 Navarra', '01 Araba/Álava', '48 Bizkaia', '20 Gipuzkoa',
       '26 Rioja, La'], dtype=object)

In [102]:
# Crear una nueva columna con el codigo de Provincia
df_comunidades['cod_provincia'] = (
    df_comunidades['Provincias']
    .str.strip()
    .str[:2]
)


# Crear una nueva columna con el nombre de la Provincia
df_comunidades['nombre_provincia'] = (
    df_comunidades['Provincias']
    .str.strip()
    .str[3:]
    .str.strip()
)

df_comunidades

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia: Nivel 1,Residencia: Nivel 2,Periodo,Total,cod_comunidad,nombre_comunidad,cod_provincia,nombre_provincia
0,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M04,10211051,NaN,NaN,NaN,NaN
1,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M03,8270160,NaN,NaN,NaN,NaN
2,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M02,6594807,NaN,NaN,NaN,NaN
3,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M01,5948424,NaN,NaN,NaN,NaN
4,Total Nacional,NaN,NaN,Viajero,Total,NaN,2025M12,6937385,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
137755,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M05,1305,19,Melilla,NaN,NaN
137756,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M04,1668,19,Melilla,NaN,NaN
137757,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M03,1673,19,Melilla,NaN,NaN
137758,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M02,1487,19,Melilla,NaN,NaN


In [103]:
# Extraer el año y el mes

# 1.- Crear 2 columnas nuevas usando el separador "M"
df_comunidades["año"] = df_comunidades["Periodo"].str.split('M').str[0]
df_comunidades["mes"] = df_comunidades["Periodo"].str.split('M').str[1]

# 2.- Convertirlas a tipo integer
df_comunidades["mes"] = pd.to_numeric(df_comunidades["mes"], errors='coerce')
df_comunidades["año"] = df_comunidades["año"].astype(int)

In [104]:
df_comunidades

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia: Nivel 1,Residencia: Nivel 2,Periodo,Total,cod_comunidad,nombre_comunidad,cod_provincia,nombre_provincia,año,mes
0,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M04,10211051,NaN,NaN,NaN,NaN,2026,4
1,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M03,8270160,NaN,NaN,NaN,NaN,2026,3
2,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M02,6594807,NaN,NaN,NaN,NaN,2026,2
3,Total Nacional,NaN,NaN,Viajero,Total,NaN,2026M01,5948424,NaN,NaN,NaN,NaN,2026,1
4,Total Nacional,NaN,NaN,Viajero,Total,NaN,2025M12,6937385,NaN,NaN,NaN,NaN,2025,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137755,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M05,1305,19,Melilla,NaN,NaN,1999,5
137756,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M04,1668,19,Melilla,NaN,NaN,1999,4
137757,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M03,1673,19,Melilla,NaN,NaN,1999,3
137758,Total Nacional,19 Melilla,NaN,Pernoctaciones,Total,Residentes en el Extranjero,1999M02,1487,19,Melilla,NaN,NaN,1999,2


In [105]:
# Comprobar los valores que coge la columna "Residencia: Nivel 1"
df_comunidades['Residencia: Nivel 1'].unique()

array(['Total'], dtype=object)

In [106]:
# No aporta nada esta columna, por lo que la borro
df_comunidades.drop(columns=["Residencia: Nivel 1"], inplace=True)

In [107]:
df_comunidades

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia: Nivel 2,Periodo,Total,cod_comunidad,nombre_comunidad,cod_provincia,nombre_provincia,año,mes
0,Total Nacional,NaN,NaN,Viajero,NaN,2026M04,10211051,NaN,NaN,NaN,NaN,2026,4
1,Total Nacional,NaN,NaN,Viajero,NaN,2026M03,8270160,NaN,NaN,NaN,NaN,2026,3
2,Total Nacional,NaN,NaN,Viajero,NaN,2026M02,6594807,NaN,NaN,NaN,NaN,2026,2
3,Total Nacional,NaN,NaN,Viajero,NaN,2026M01,5948424,NaN,NaN,NaN,NaN,2026,1
4,Total Nacional,NaN,NaN,Viajero,NaN,2025M12,6937385,NaN,NaN,NaN,NaN,2025,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137755,Total Nacional,19 Melilla,NaN,Pernoctaciones,Residentes en el Extranjero,1999M05,1305,19,Melilla,NaN,NaN,1999,5
137756,Total Nacional,19 Melilla,NaN,Pernoctaciones,Residentes en el Extranjero,1999M04,1668,19,Melilla,NaN,NaN,1999,4
137757,Total Nacional,19 Melilla,NaN,Pernoctaciones,Residentes en el Extranjero,1999M03,1673,19,Melilla,NaN,NaN,1999,3
137758,Total Nacional,19 Melilla,NaN,Pernoctaciones,Residentes en el Extranjero,1999M02,1487,19,Melilla,NaN,NaN,1999,2


## 01.3.- Filtar 5 años ultimos

In [108]:
df_comunidades['año'].unique()

array([2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016,
       2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005,
       2004, 2003, 2002, 2001, 2000, 1999])

In [109]:
# Crear un df nuevo que contenga solo los registros de los años desde el 2021 al 2025 enteros
df_comunidades_limpio = df_comunidades[
    (df_comunidades['año'] >= 2021) &
    (df_comunidades['año'] <= 2025)
].copy()

df_comunidades_limpio['año'].unique()

array([2025, 2024, 2023, 2022, 2021])

## 01.4.- Exportar a fichero csv

In [110]:
# Exportar df_comunidades_limpio limpio
df_comunidades_limpio.to_csv("2026_06_15_comunidades_limpio.csv", index=False)

# 02.- Viajeros y pernoctaciones según país de residencia del viajero

## 02.1.- Importar fichero csv

In [4]:
df_residencias = pd.read_csv("2026_06_15_Viajeros_pernoctaciones_según_país_de_residencia_del_viajero.csv", sep=';')

In [112]:
df_residencias

,RESIDENCIA/ORIGEN,Países,Viajeros y pernoctaciones,Periodo,Total
0,Total,NaN,Viajero,2026M04,10.211.051
1,Total,NaN,Viajero,2026M03,8.270.160
2,Total,NaN,Viajero,2026M02,6.594.807
3,Total,NaN,Viajero,2026M01,5.948.424
4,Total,NaN,Viajero,2025M12,6.937.385
...,...,...,...,...,...
20987,Asia (sin Japón),NaN,Pernoctaciones,1999M05,25.427
20988,Asia (sin Japón),NaN,Pernoctaciones,1999M04,26.878
20989,Asia (sin Japón),NaN,Pernoctaciones,1999M03,24.271
20990,Asia (sin Japón),NaN,Pernoctaciones,1999M02,18.530


## 02.2.- Transformación

In [113]:
# Tipos de datos e información general
df_residencias.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20992 entries, 0 to 20991
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   RESIDENCIA/ORIGEN          20992 non-null  object
 1   Países                     17712 non-null  object
 2   Viajeros y pernoctaciones  20992 non-null  object
 3   Periodo                    20992 non-null  object
 4   Total                      19802 non-null  object
dtypes: object(5)
memory usage: 820.1+ KB


In [114]:
# Convertir columna Total de object a numerica

# 1.- Quitar el punto (.) como separador de miles
df_residencias["Total"] = df_residencias["Total"].str.replace(".", "", regex=False)

# 2.- Reemplazar el "0" por NaN
df_residencias["Total"] = df_residencias["Total"].replace({"0": np.nan})

# 3.- Convertir a tipo integer
df_residencias["Total"] = pd.to_numeric(df_residencias["Total"]).astype("Int64")

df_residencias.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20992 entries, 0 to 20991
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   RESIDENCIA/ORIGEN          20992 non-null  object
 1   Países                     17712 non-null  object
 2   Viajeros y pernoctaciones  20992 non-null  object
 3   Periodo                    20992 non-null  object
 4   Total                      19580 non-null  Int64 
dtypes: Int64(1), object(4)
memory usage: 840.6+ KB


In [115]:
df_residencias

,RESIDENCIA/ORIGEN,Países,Viajeros y pernoctaciones,Periodo,Total
0,Total,NaN,Viajero,2026M04,10211051
1,Total,NaN,Viajero,2026M03,8270160
2,Total,NaN,Viajero,2026M02,6594807
3,Total,NaN,Viajero,2026M01,5948424
4,Total,NaN,Viajero,2025M12,6937385
...,...,...,...,...,...
20987,Asia (sin Japón),NaN,Pernoctaciones,1999M05,25427
20988,Asia (sin Japón),NaN,Pernoctaciones,1999M04,26878
20989,Asia (sin Japón),NaN,Pernoctaciones,1999M03,24271
20990,Asia (sin Japón),NaN,Pernoctaciones,1999M02,18530


In [116]:
# Extraer el año y el mes

# 1.- Crear 2 columnas nuevas usando el separador "M"
df_residencias["año"] = df_residencias["Periodo"].str.split('M').str[0]
df_residencias["mes"] = df_residencias["Periodo"].str.split('M').str[1]

# 2.- Convertirlas a tipo integer
df_residencias["mes"] = pd.to_numeric(df_residencias["mes"], errors='coerce')
df_residencias["año"] = df_residencias["año"].astype(int)

df_residencias

,RESIDENCIA/ORIGEN,Países,Viajeros y pernoctaciones,Periodo,Total,año,mes
0,Total,NaN,Viajero,2026M04,10211051,2026,4
1,Total,NaN,Viajero,2026M03,8270160,2026,3
2,Total,NaN,Viajero,2026M02,6594807,2026,2
3,Total,NaN,Viajero,2026M01,5948424,2026,1
4,Total,NaN,Viajero,2025M12,6937385,2025,12
...,...,...,...,...,...,...,...
20987,Asia (sin Japón),NaN,Pernoctaciones,1999M05,25427,1999,5
20988,Asia (sin Japón),NaN,Pernoctaciones,1999M04,26878,1999,4
20989,Asia (sin Japón),NaN,Pernoctaciones,1999M03,24271,1999,3
20990,Asia (sin Japón),NaN,Pernoctaciones,1999M02,18530,1999,2


## 02.3.- Filtar 5 años ultimos

In [117]:
df_residencias['año'].unique()

array([2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016,
       2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005,
       2004, 2003, 2002, 2001, 2000, 1999])

In [118]:
# Crear un df nuevo que contenga solo los registros de los años desde el 2021 al 2025 enteros
df_residencias_limpio = df_residencias[
    (df_residencias['año'] >= 2021) &
    (df_residencias['año'] <= 2025)
].copy()

df_residencias_limpio['año'].unique()

array([2025, 2024, 2023, 2022, 2021])

In [119]:
df_residencias_limpio

,RESIDENCIA/ORIGEN,Países,Viajeros y pernoctaciones,Periodo,Total,año,mes
4,Total,NaN,Viajero,2025M12,6937385,2025,12
5,Total,NaN,Viajero,2025M11,7413495,2025,11
6,Total,NaN,Viajero,2025M10,11041430,2025,10
7,Total,NaN,Viajero,2025M09,12050972,2025,9
8,Total,NaN,Viajero,2025M08,13828672,2025,8
...,...,...,...,...,...,...,...
20723,Asia (sin Japón),NaN,Pernoctaciones,2021M05,<NA>,2021,5
20724,Asia (sin Japón),NaN,Pernoctaciones,2021M04,<NA>,2021,4
20725,Asia (sin Japón),NaN,Pernoctaciones,2021M03,<NA>,2021,3
20726,Asia (sin Japón),NaN,Pernoctaciones,2021M02,<NA>,2021,2


## 02.4.- Exportar a fichero csv

In [120]:
# Exportar df_comunidades_limpio limpio
df_residencias_limpio.to_csv("2026_06_15_residencias_limpio.csv", index=False)

# 03.- Viajeros y pernoctaciones por puntos turísticos

## 03.1.- Importar fichero csv

In [6]:
df_ciudades = pd.read_csv("2026_06_15_Viajeros_pernoctaciones_por_puntos_turísticos.csv", sep=';')

In [122]:
df_ciudades

,Puntos turísticos,Viajeros y pernoctaciones,Residencia,Periodo,Total
0,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M04,21.012
1,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M03,20.189
2,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M02,18.573
3,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M01,18.376
4,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2025M12,23.737
...,...,...,...,...,...
141307,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M05,20.698
141308,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M04,20.003
141309,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M03,21.156
141310,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M02,13.372


## 03.2.- Transformación

In [123]:
# Tipos de datos e información general
df_ciudades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141312 entries, 0 to 141311
Data columns (total 5 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Puntos turísticos          141312 non-null  object
 1   Viajeros y pernoctaciones  141312 non-null  object
 2   Residencia                 141312 non-null  object
 3   Periodo                    141312 non-null  object
 4   Total                      106730 non-null  object
dtypes: object(5)
memory usage: 5.4+ MB


In [124]:
# Convertir columna Total de object a numerica

# 1.- Quitar el punto (.) como separador de miles
df_ciudades["Total"] = df_ciudades["Total"].str.replace(".", "", regex=False)

# 2.- Reemplazar el "0" por NaN
df_ciudades["Total"] = df_ciudades["Total"].replace({"0": np.nan})

# 3.- Convertir a tipo integer
df_ciudades["Total"] = pd.to_numeric(df_ciudades["Total"]).astype("Int64")

df_ciudades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141312 entries, 0 to 141311
Data columns (total 5 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Puntos turísticos          141312 non-null  object
 1   Viajeros y pernoctaciones  141312 non-null  object
 2   Residencia                 141312 non-null  object
 3   Periodo                    141312 non-null  object
 4   Total                      95947 non-null   Int64 
dtypes: Int64(1), object(4)
memory usage: 5.5+ MB


In [125]:
df_ciudades

,Puntos turísticos,Viajeros y pernoctaciones,Residencia,Periodo,Total
0,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M04,21012
1,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M03,20189
2,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M02,18573
3,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M01,18376
4,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2025M12,23737
...,...,...,...,...,...
141307,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M05,20698
141308,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M04,20003
141309,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M03,21156
141310,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M02,13372


In [126]:
# Extraer el año y el mes

# 1.- Crear 2 columnas nuevas usando el separador "M"
df_ciudades["año"] = df_ciudades["Periodo"].str.split('M').str[0]
df_ciudades["mes"] = df_ciudades["Periodo"].str.split('M').str[1]

# 2.- Convertirlas a tipo integer
df_ciudades["mes"] = pd.to_numeric(df_ciudades["mes"], errors='coerce')
df_ciudades["año"] = df_ciudades["año"].astype(int)

df_ciudades

,Puntos turísticos,Viajeros y pernoctaciones,Residencia,Periodo,Total,año,mes
0,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M04,21012,2026,4
1,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M03,20189,2026,3
2,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M02,18573,2026,2
3,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M01,18376,2026,1
4,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2025M12,23737,2025,12
...,...,...,...,...,...,...,...
141307,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M05,20698,2005,5
141308,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M04,20003,2005,4
141309,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M03,21156,2005,3
141310,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M02,13372,2005,2


In [ ]:
df_ciudades['Puntos turísticos'].unique()

array(['01059 Vitoria-Gasteiz', '02003 Albacete',
       '03014 Alacant/Alicante', '03018 Altea', '03031 Benidorm',
       '03063 Dénia', '03065 Elx/Elche', '03133 Torrevieja',
       '04013 Almería', '04032 Carboneras', '04064 Mojácar',
       '04066 Níjar', '04079 Roquetas de Mar', '04902 Ejido, El',
       '05019 Ávila', '06015 Badajoz', '06083 Mérida', '07003 Alcúdia',
       '07011 Calvià', '07014 Capdepera', '07015 Ciutadella de Menorca',
       '07024 Formentera', '07026 Eivissa', '07039 Muro', '07040 Palma',
       '07042 Pollença', '07046 Sant Antoni de Portmany',
       '07048 Sant Josep de sa Talaia',
       '07051 Sant Llorenç des Cardassar', '07057 Santanyí',
       '07061 Sóller', '08019 Barcelona', '08056 Castelldefels',
       "08101 Hospitalet de Llobregat, L'", '08270 Sitges',
       '09059 Burgos', '10037 Cáceres', '10148 Plasencia',
       '10195 Trujillo', '11004 Algeciras', '11006 Arcos de la Frontera',
       '11012 Cádiz', '11014 Conil de la Frontera',
       '1

In [127]:
df_ciudades['Puntos turísticos'].unique()

array(['01059 Vitoria-Gasteiz', '02003 Albacete',
       '03014 Alacant/Alicante', '03018 Altea', '03031 Benidorm',
       '03063 Dénia', '03065 Elx/Elche', '03133 Torrevieja',
       '04013 Almería', '04032 Carboneras', '04064 Mojácar',
       '04066 Níjar', '04079 Roquetas de Mar', '04902 Ejido, El',
       '05019 Ávila', '06015 Badajoz', '06083 Mérida', '07003 Alcúdia',
       '07011 Calvià', '07014 Capdepera', '07015 Ciutadella de Menorca',
       '07024 Formentera', '07026 Eivissa', '07039 Muro', '07040 Palma',
       '07042 Pollença', '07046 Sant Antoni de Portmany',
       '07048 Sant Josep de sa Talaia',
       '07051 Sant Llorenç des Cardassar', '07057 Santanyí',
       '07061 Sóller', '08019 Barcelona', '08056 Castelldefels',
       "08101 Hospitalet de Llobregat, L'", '08270 Sitges',
       '09059 Burgos', '10037 Cáceres', '10148 Plasencia',
       '10195 Trujillo', '11004 Algeciras', '11006 Arcos de la Frontera',
       '11012 Cádiz', '11014 Conil de la Frontera',
       '1

In [128]:
# Crear una nueva columna con el codigo postal de la Ciudad
df_ciudades['cod_postal_ciudad'] = (
    df_ciudades['Puntos turísticos']
    .str.strip()
    .str[:5]
)


# Crear una nueva columna con el nombre de la Ciudad
df_ciudades['nombre_ciudad'] = (
    df_ciudades['Puntos turísticos']
    .str.strip()
    .str[6:]
    .str.strip()
)

df_ciudades

,Puntos turísticos,Viajeros y pernoctaciones,Residencia,Periodo,Total,año,mes,cod_postal_ciudad,nombre_ciudad
0,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M04,21012,2026,4,01059,Vitoria-Gasteiz
1,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M03,20189,2026,3,01059,Vitoria-Gasteiz
2,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M02,18573,2026,2,01059,Vitoria-Gasteiz
3,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2026M01,18376,2026,1,01059,Vitoria-Gasteiz
4,01059 Vitoria-Gasteiz,Viajero,Residentes en España,2025M12,23737,2025,12,01059,Vitoria-Gasteiz
...,...,...,...,...,...,...,...,...,...
141307,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M05,20698,2005,5,50297,Zaragoza
141308,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M04,20003,2005,4,50297,Zaragoza
141309,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M03,21156,2005,3,50297,Zaragoza
141310,50297 Zaragoza,Pernoctaciones,Residentes en el Extranjero,2005M02,13372,2005,2,50297,Zaragoza


## 03.3.- Filtar 5 años ultimos

In [129]:
df_ciudades['año'].unique()

array([2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016,
       2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005])

In [130]:
# Crear un df nuevo que contenga solo los registros de los años desde el 2021 al 2025 enteros
df_ciudades_limpio = df_ciudades[
    (df_ciudades['año'] >= 2021) &
    (df_ciudades['año'] <= 2025)
].copy()

df_ciudades_limpio['año'].unique()

array([2025, 2024, 2023, 2022, 2021])

## 03.4.- Exportar a fichero csv

In [131]:
# Exportar df_comunidades_limpio limpio
df_ciudades_limpio.to_csv("2026_06_15_ciudades_limpio.csv", index=False)

# 04.- Viajeros, pernoctaciones por tipo de alojamiento por comunidades y ciudades autónomas

## 04.1.- Importar fichero csv

In [10]:
df_alojamientos = pd.read_csv("2026_06_15_Viajeros_pernoctaciones_por_tipo_alojamiento_por_comunidades_y_ciudades_autónomas.csv", sep=';')

In [11]:
df_alojamientos

,Tipo de alojamiento,Total Nacional,Comunidades y Ciudades Autónomas,Residencia: Nivel 1,Residencia: Nivel 2,Viajeros y pernoctaciones,Periodo,Total
0,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M04,10.211.051
1,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M03,8.270.160
2,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M02,6.594.807
3,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M01,5.948.424
4,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2025M12,6.937.385
...,...,...,...,...,...,...,...,...
182395,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M05,NaN
182396,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M04,NaN
182397,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M03,NaN
182398,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M02,NaN


## 04.2.- Transformación

In [12]:
# Tipos de datos e información general
df_alojamientos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182400 entries, 0 to 182399
Data columns (total 8 columns):
 #   Column                            Non-Null Count   Dtype 
---  ------                            --------------   ----- 
 0   Tipo de alojamiento               182400 non-null  object
 1   Total Nacional                    182400 non-null  object
 2   Comunidades y Ciudades Autónomas  173280 non-null  object
 3   Residencia: Nivel 1               182400 non-null  object
 4   Residencia: Nivel 2               121600 non-null  object
 5   Viajeros y pernoctaciones         182400 non-null  object
 6   Periodo                           182400 non-null  object
 7   Total                             152560 non-null  object
dtypes: object(8)
memory usage: 11.1+ MB


In [13]:
# Convertir columna Total de object a numerica

# 1.- Quitar el punto (.) como separador de miles
df_alojamientos["Total"] = df_alojamientos["Total"].str.replace(".", "", regex=False)

# 2.- Reemplazar el "0" por NaN
df_alojamientos["Total"] = df_alojamientos["Total"].replace({"0": np.nan})

# 3.- Convertir a tipo integer
df_alojamientos["Total"] = pd.to_numeric(df_alojamientos["Total"]).astype("Int64")

df_alojamientos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182400 entries, 0 to 182399
Data columns (total 8 columns):
 #   Column                            Non-Null Count   Dtype 
---  ------                            --------------   ----- 
 0   Tipo de alojamiento               182400 non-null  object
 1   Total Nacional                    182400 non-null  object
 2   Comunidades y Ciudades Autónomas  173280 non-null  object
 3   Residencia: Nivel 1               182400 non-null  object
 4   Residencia: Nivel 2               121600 non-null  object
 5   Viajeros y pernoctaciones         182400 non-null  object
 6   Periodo                           182400 non-null  object
 7   Total                             145141 non-null  Int64 
dtypes: Int64(1), object(7)
memory usage: 11.3+ MB


In [14]:
# Extraer el año y el mes

# 1.- Crear 2 columnas nuevas usando el separador "M"
df_alojamientos["año"] = df_alojamientos["Periodo"].str.split('M').str[0]
df_alojamientos["mes"] = df_alojamientos["Periodo"].str.split('M').str[1]

# 2.- Convertirlas a tipo integer
df_alojamientos["mes"] = pd.to_numeric(df_alojamientos["mes"], errors='coerce')
df_alojamientos["año"] = df_alojamientos["año"].astype(int)

df_alojamientos

,Tipo de alojamiento,Total Nacional,Comunidades y Ciudades Autónomas,Residencia: Nivel 1,Residencia: Nivel 2,Viajeros y pernoctaciones,Periodo,Total,año,mes
0,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M04,10211051,2026,4
1,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M03,8270160,2026,3
2,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M02,6594807,2026,2
3,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M01,5948424,2026,1
4,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2025M12,6937385,2025,12
...,...,...,...,...,...,...,...,...,...,...
182395,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M05,<NA>,2001,5
182396,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M04,<NA>,2001,4
182397,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M03,<NA>,2001,3
182398,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M02,<NA>,2001,2


In [15]:
df_alojamientos['Tipo de alojamiento'].unique()

array(['Encuesta de Ocupación Hotelera',
       'Encuesta de Ocupación en Campings',
       'Encuesta de Ocupación en Apartamentos Turísticos',
       'Encuesta de Ocupación en Alojamientos de Turismo Rural',
       'Encuesta de Ocupación en Albergues'], dtype=object)

In [16]:
df_alojamientos['Total Nacional'].unique()

array(['Total Nacional'], dtype=object)

In [18]:
df_alojamientos['Comunidades y Ciudades Autónomas'].unique()

array([nan, '01 Andalucía', '02 Aragón', '03 Asturias, Principado de',
       '04 Balears, Illes', '05 Canarias', '06 Cantabria',
       '07 Castilla y León', '08 Castilla - La Mancha', '09 Cataluña',
       '10 Comunitat Valenciana', '11 Extremadura', '12 Galicia',
       '13 Madrid, Comunidad de', '14 Murcia, Región de',
       '15 Navarra, Comunidad Foral de', '16 País Vasco', '17 Rioja, La',
       '18 Ceuta', '19 Melilla'], dtype=object)

In [19]:
# Crear una nueva columna con el codigo de comunidad
df_alojamientos['cod_comunidad'] = (
    df_alojamientos['Comunidades y Ciudades Autónomas']
    .str.strip()
    .str[:2]
)


# Crear una nueva columna con el nombre de la comunidad
df_alojamientos['nombre_comunidad'] = (
    df_alojamientos['Comunidades y Ciudades Autónomas']
    .str.strip()
    .str[3:]
    .str.strip()
)

df_alojamientos

,Tipo de alojamiento,Total Nacional,Comunidades y Ciudades Autónomas,Residencia: Nivel 1,Residencia: Nivel 2,Viajeros y pernoctaciones,Periodo,Total,año,mes,cod_comunidad,nombre_comunidad
0,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M04,10211051,2026,4,NaN,NaN
1,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M03,8270160,2026,3,NaN,NaN
2,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M02,6594807,2026,2,NaN,NaN
3,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2026M01,5948424,2026,1,NaN,NaN
4,Encuesta de Ocupación Hotelera,Total Nacional,NaN,Total,NaN,Viajero,2025M12,6937385,2025,12,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
182395,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M05,<NA>,2001,5,19,Melilla
182396,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M04,<NA>,2001,4,19,Melilla
182397,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M03,<NA>,2001,3,19,Melilla
182398,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Total,Residentes en el Extranjero,Pernoctaciones,2001M02,<NA>,2001,2,19,Melilla


In [20]:
df_alojamientos['Residencia: Nivel 1'].unique()

array(['Total'], dtype=object)

In [23]:
# No aporta nada esta columna, por lo que la borro
df_alojamientos.drop(columns=["Residencia: Nivel 1"], inplace=True)

In [21]:
df_alojamientos['Residencia: Nivel 2'].unique()

array([nan, 'Residentes en España', 'Residentes en el Extranjero'],
      dtype=object)

In [22]:
df_alojamientos['Viajeros y pernoctaciones'].unique()

array(['Viajero', 'Pernoctaciones'], dtype=object)

In [25]:
df_alojamientos

,Tipo de alojamiento,Total Nacional,Comunidades y Ciudades Autónomas,Residencia: Nivel 2,Viajeros y pernoctaciones,Periodo,Total,año,mes,cod_comunidad,nombre_comunidad
0,Encuesta de Ocupación Hotelera,Total Nacional,NaN,NaN,Viajero,2026M04,10211051,2026,4,NaN,NaN
1,Encuesta de Ocupación Hotelera,Total Nacional,NaN,NaN,Viajero,2026M03,8270160,2026,3,NaN,NaN
2,Encuesta de Ocupación Hotelera,Total Nacional,NaN,NaN,Viajero,2026M02,6594807,2026,2,NaN,NaN
3,Encuesta de Ocupación Hotelera,Total Nacional,NaN,NaN,Viajero,2026M01,5948424,2026,1,NaN,NaN
4,Encuesta de Ocupación Hotelera,Total Nacional,NaN,NaN,Viajero,2025M12,6937385,2025,12,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
182395,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Residentes en el Extranjero,Pernoctaciones,2001M05,<NA>,2001,5,19,Melilla
182396,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Residentes en el Extranjero,Pernoctaciones,2001M04,<NA>,2001,4,19,Melilla
182397,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Residentes en el Extranjero,Pernoctaciones,2001M03,<NA>,2001,3,19,Melilla
182398,Encuesta de Ocupación en Albergues,Total Nacional,19 Melilla,Residentes en el Extranjero,Pernoctaciones,2001M02,<NA>,2001,2,19,Melilla


## 04.3.- Filtar 5 años ultimos

In [26]:
df_alojamientos['año'].unique()

array([2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016,
       2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005,
       2004, 2003, 2002, 2001])

In [27]:
# Crear un df nuevo que contenga solo los registros de los años desde el 2021 al 2025 enteros
df_alojamientos_limpio = df_alojamientos[
    (df_alojamientos['año'] >= 2021) &
    (df_alojamientos['año'] <= 2025)
].copy()

df_alojamientos_limpio['año'].unique()

array([2025, 2024, 2023, 2022, 2021])

## 04.4.- Exportar a fichero csv

In [28]:
# Exportar df_comunidades_limpio limpio
df_alojamientos_limpio.to_csv("2026_06_15_alojamientos_limpio.csv", index=False)